# 15. Test ablation: class-specific thresholds

Порог каждого класса уже зафиксирован на validation. Test используется один раз только для оценки ablation; исходные глобальные test-артефакты не перезаписываются.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import runpy

PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
EXPERIMENT_CONFIG = PROJECT_DIR / 'configs/experiments/span_ner_corrected_v1.yaml'
OUTPUT_DIR = PROJECT_DIR / 'results/span_ner_corrected_v1/seed_42'
CHECKPOINT = OUTPUT_DIR / 'checkpoints/best'
GLOBAL_CALIBRATION = OUTPUT_DIR / 'threshold_calibration.json'
CLASS_CALIBRATION = OUTPUT_DIR / 'class_threshold_calibration.json'
GLOBAL_TEST_METRICS = OUTPUT_DIR / 'test_metrics.json'
BOOTSTRAP = PROJECT_DIR / 'colab_bootstrap.py'

for required_path in (EXPERIMENT_CONFIG, CHECKPOINT, GLOBAL_CALIBRATION, CLASS_CALIBRATION, BOOTSTRAP):
    if not required_path.exists():
        raise FileNotFoundError(f'Не найден {required_path}. Выполните предыдущие Span notebooks.')
global_report = json.loads(GLOBAL_CALIBRATION.read_text(encoding='utf-8'))
class_report = json.loads(CLASS_CALIBRATION.read_text(encoding='utf-8'))
GLOBAL_THRESHOLD = float(global_report['best_threshold'])
CLASS_THRESHOLDS = {key: float(value) for key, value in class_report['class_thresholds'].items()}
bootstrap_project = runpy.run_path(str(BOOTSTRAP))['bootstrap_project']
bootstrap_project(PROJECT_DIR)

In [ ]:
from rurebus_ie.training import test_span_ner_experiment

result = test_span_ner_experiment(
    EXPERIMENT_CONFIG,
    project_root=PROJECT_DIR,
    checkpoint_dir=CHECKPOINT,
    confidence_threshold_override=GLOBAL_THRESHOLD,
    class_thresholds_override=CLASS_THRESHOLDS,
    artifact_prefix='class_thresholds_test',
)
print(f'Class-specific test micro-F1: {result.metrics.micro_f1:.4f}')
print(f'Class-specific test macro-F1: {result.metrics.macro_f1:.4f}')

In [ ]:
import pandas as pd

class_metrics = {
    'precision': result.metrics.precision,
    'recall': result.metrics.recall,
    'micro_f1': result.metrics.micro_f1,
    'macro_f1': result.metrics.macro_f1,
}
rows = {'class_specific': class_metrics}
if GLOBAL_TEST_METRICS.is_file():
    global_test = json.loads(GLOBAL_TEST_METRICS.read_text(encoding='utf-8'))
    rows['global_0.82'] = {key: global_test[key] for key in class_metrics}
display(pd.DataFrame(rows).T)
display(pd.DataFrame(result.metrics.per_class).T.sort_values('f1'))
print('Метрики ablation:', OUTPUT_DIR / 'class_thresholds_test_metrics.json')
print('Глобальные test-метрики не изменены.')